<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.24em;text-transform:uppercase;color:#A20025;margin:0 0 12px;">NeurIPS 2026 Education Track&nbsp;&middot;&nbsp;Generate, then verify</div>

<h1 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:33px;line-height:1.12;margin:0 0 16px;color:#1A1416;letter-spacing:-.01em;">Worked solutions 1 to 3. Sampling budget and controls</h1>

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:0 0 4px;">

<em style="color:#6B6461;font-size:14.5px;">Full working for X1, X2 and X3, with the reasoning behind each answer. Open this after you have an answer of your own.</em>

Every cell here runs, and every number printed is the number the exercise expects.
Where a task has a written part, the prose below gives an answer that would earn
full marks and says what makes it earn them, because the wording is usually the
graded thing rather than the number.

A note on tolerances before we start. The energy distances in X1 and X2 are seeded,
so a given run reproduces exactly on the same machine, but the exercise deliberately
does not pin the reference draw to the five the shipped sweep averaged over. A
single-run value therefore sits near its seed-mean rather than on it, and at the
smaller budgets it can sit a third away. The two energy distances that are
self-checked carry a 35 percent window, sized from the measured seed-to-seed spread;
the crossing budget, which is identical under every seed, is the one thing graded
exactly. Chasing a digit-for-digit match on the rest would teach the wrong habit: a
conclusion has to survive the seed, and if it does not, it was never a conclusion.

In [1]:
# Path anchor and toolkit import. This is the only setup cell in the notebook.
#
# The bundle looks like this:
#     NEURIPS2026_EDU_GENERATE_VERIFY/
#         gv_toolkit.py          <- the shared toolkit
#         data/                 <- every cached file the exercises read
#         exercises/            <- you are here
#         solutions/
# so the root is one level up from this notebook, and data/ hangs off the root.
# Resolving it positionally means the same cell works locally, on Colab, and in a
# grader's batch run, with no absolute path anywhere.
import sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name in {"exercises", "solutions"} else HERE
sys.path.insert(0, str(ROOT))

import gv_toolkit as gv

DATA = gv.DATA
print("bundle root:", ROOT.name, "| cached inputs:", len(gv.REQUIRED_FILES) + len(gv.SUPPORTING_FILES))

bundle root: NEURIPS2026_EDU_GENERATE_VERIFY | cached inputs: 25


In [2]:
import numpy as np
import torch

torch.set_num_threads(4)
ddpm_net = gv.load_pinned("toy_ddpm_trained.pt")
fm_net = gv.load_pinned("toy_fm_trained.pt")
BAND, N_EVAL, SEED = 0.01, 2000, 0

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;1</span>X1. Find the step budget where the sampler breaks</h2>

The measurement is three lines. The interesting part is what the numbers do between
budgets, so read the column before reading the answer.

In [3]:
BUDGETS = [200, 32, 16, 8, 4]
reference = gv.sample_target(N_EVAL, seed=1000)

measured = {K: gv.energy_distance(
    gv.ddpm_sample(ddpm_net, N_EVAL, steps=K, seed=SEED).numpy(), reference)
    for K in BUDGETS}

for K in BUDGETS:
    flag = "under the band" if measured[K] < BAND else ""
    print(f"  {K:>3} steps   {measured[K]:.6f}   {flag}")

crossing_budget = min(K for K in BUDGETS if measured[K] < BAND)
print(f"\ncrossing budget: {crossing_budget}")
print(f"ratio, 8 steps against 200 steps: {measured[8] / measured[200]:.0f}x worse")

  200 steps   0.001686   under the band
   32 steps   0.006445   under the band
   16 steps   0.012888   
    8 steps   0.037125   
    4 steps   1.203524   

crossing budget: 32
ratio, 8 steps against 200 steps: 22x worse


**The answer.** Ancestral DDPM crosses the 0.01 band at 32 steps.

Keep two sets of numbers distinct here when marking. The cell above measures
a **single seeded run** against one reference draw: 0.001686 at 200 steps, 0.006445
at 32, 0.012888 at 16, 0.037125 at 8, and 1.203524 at 4. The shipped sweep reports
**five-seed means** over five independent reference draws: 0.00177, 0.00475, 0.01398,
0.0407 and 1.21777 at the same five budgets.

The two agree on everything that matters and differ by up to a third on the
individual figures. At 32 steps the single run reads 0.006445 against a seed-mean of
0.00475, a 36 percent gap; at 8 steps it reads 0.037125 against 0.0407, a 9 percent
gap. That spread is why the crossing budget is the graded answer: it is 32 under
every seed in the shipped sweep, so it can be marked exactly, while an individual
energy distance cannot.

Only two energy distances are self-checked, at 200 and 8 steps, and each carries a 35
percent window around the seed-mean. That window is set from the measured seed spread
rather than picked round: across the five shipped seeds the 200-step value ranges from
0.001396 to 0.001963 and the 8-step value from 0.035893 to 0.051148, so a 20 percent
window would have failed a correct answer on some seeds. The single run above sits
inside both windows (0.001686 against a seed-mean of 0.00177, and 0.037125 against
0.0407). The 32-step value is not checked at all: its seed spread reaches 36 percent
and there is nothing to gain from grading it, since the crossing budget already
carries that part of the answer.

**Why 16 is wrong.** It is tempting to answer 16 because 0.0140 looks close to 0.01.
It is 40 percent above the band. A threshold you cross when you squint is not a
threshold, and the whole point of stating a band up front is that you then have to
respect it.

**What the scatter plots would have told you.** Not this. Look at the five panels
again: 200, 32 and 16 steps are visually indistinguishable, all three showing two
well-formed moons. The degradation only becomes obvious at 8 steps, and by 4 steps the
sample is unrecognizable. Your eye puts the failure between 8 and 4. The metric puts
it between 32 and 16. A grader accepting your sentence for part three is looking for
exactly that observation: the eye saturates a full factor of two to four earlier than
the metric does, so a figure showing three panels that look fine is compatible with a
sampler that has already left the quality band.

**The pedagogy of the prediction step.** Most learners predict a smooth degradation:
each halving of the budget costs a bit more quality. What the column shows
is a knee. Between 200 and 32 the metric moves by a factor of under 3 across a
sixfold budget cut, and between 16 and 8 it moves by a factor of 3 across a single
halving. Then between 8 and 4 it moves by a factor of 30. The prediction is graded
for having been made, not for being right, because the recorded surprise is what
makes the shape memorable.

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;2</span>X2. Two ways to be fooled by a validator</h2>

Part one establishes the floor of the metric. Part two is the control, and it is the
task in this notebook that transfers furthest.

In [4]:
# Part one. The two-sample floor: both clouds drawn from the target itself.
floor_values = [
    gv.energy_distance(gv.sample_target(N_EVAL, seed=2000 + i),
                        gv.sample_target(N_EVAL, seed=3000 + i))
    for i in range(5)
]
two_sample_floor = float(np.mean(floor_values))
print("per-pair:", [round(v, 6) for v in floor_values])
print(f"mean floor: {two_sample_floor:.6f}")
print(f"the 0.01 band sits {BAND / two_sample_floor:.0f}x above it")

per-pair: [0.000153, 0.000192, 0.00014, 0.000179, 0.000187]
mean floor: 0.000170
the 0.01 band sits 59x above it


**Part one.** Two independent draws of 2000 points from the same distribution give an
energy distance around 0.0001 to 0.0002. Neither cloud came from a model, so that is
not a quality measurement; it is the metric's own noise at this sample size. It falls
as the sample size grows and it never reaches zero for finite samples.

The 0.01 band sits fifty to a hundred times above that noise floor, making it a
generous bar. Clearing 0.01 confirms that the sampler captures the gross shape of
the two crescents, while still sitting two orders of magnitude above finite-sample
noise. Claiming two samplers were indistinguishable would require a threshold near
the floor, where sample size dominates the comparison.

In [5]:
# Part two. The eta sweep holds the sampler family fixed and moves only the noise.
eta_sweep = gv.read_metrics_csv(DATA / "toy_eta_sweep.csv")
eta_mean = eta_sweep.groupby("eta").energy_distance.mean()
print(eta_mean.round(5).to_string())

noise_removal_factor = float(eta_mean.loc[0.0] / eta_mean.loc[1.0])
print(f"\nremoving the sampling noise entirely costs {noise_removal_factor:.1f}x")

sweep = gv.read_metrics_csv(DATA / "toy_sampler_sweep.csv")
mean_energy = sweep.groupby(["sampler", "steps"]).energy_distance.mean().unstack()
print()
print("at the matched 8-step budget:")
for s in ["fm_euler", "ddpm_ancestral", "ddim_eta0_x0clip", "ddim_eta0"]:
    v = mean_energy.loc[s, 8]
    print(f"  {s:<18s} {v:.5f}   ({v / mean_energy.loc['fm_euler', 8]:5.1f}x "
          "flow matching)")

eta
0.0    0.84861
0.2    0.78284
0.4    0.63052
0.6    0.54207
0.8    0.53184
1.0    0.04072

removing the sampling noise entirely costs 20.8x

at the matched 8-step budget:
  fm_euler           0.00998   (  1.0x flow matching)
  ddpm_ancestral     0.04072   (  4.1x flow matching)
  ddim_eta0_x0clip   0.05953   (  6.0x flow matching)
  ddim_eta0          0.84861   ( 85.0x flow matching)


**Part two, the confound: sampling stochasticity.** The flow-matching sampler is a
deterministic ODE solve. Ancestral DDPM draws fresh Gaussian noise at every reverse
step. So when you compare them at a matched step budget, you are comparing two things
at once: the shape of the path, and whether the sampler is stochastic. The path
explanation may be correct, but the experiment as posed cannot support it, because
the two candidate causes are perfectly confounded.

**Holding it fixed.** The eta dial does exactly that. At eta of 1 the DDIM update is
mathematically identical to the ancestral one (verified numerically to float64
precision, not just asserted). At eta of 0 it is fully deterministic. Nothing
else changes: same network, same ladder, same 8-step budget, same seeds. Sweeping eta
therefore isolates the stochasticity and leaves everything else alone.

**And the gap widens.** This is the result that surprises people. Deterministic DDIM
at eta of 0 never reaches the band at any budget the sweep tested, sitting at 0.0732
even with 200 steps, while ancestral DDPM crosses at 32. At the matched 8-step budget
the deterministic sampler is 0.849 against the stochastic sampler's 0.041, a factor
of about 21. So when you hold the sampler family fixed at the deterministic end,
flow matching's advantage over the diffusion sampler does not narrow; it widens
enormously.

**What that means for the original explanation.** Path straightness cannot carry the
whole gap: within the diffusion family, holding the path fixed and stripping the
noise worsens energy distance by 20.8x. On this toy, per-step Gaussian noise damps
prediction error: the deterministic DDIM update estimates the denoised point by
dividing by the square root of alpha-bar, which is tiny at the high-noise end of an
8-step ladder and amplifies any error in the predicted noise. The ancestral update
divides by the square root of alpha instead and stays bounded. Clipping that
runaway `x0` estimate recovers most of the loss, so the `ddim_eta0_x0clip` arm of
the sweep crosses the band at 64 steps.

**A learner who predicted "narrows" has still met the objective.** The objective is
naming the confound and building the control. Getting the direction wrong and then
measuring it is the correct scientific move. A learner who reports "narrows" after
running the sweep has misread the table, which is a different problem.

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;3</span>X3. One network, two jobs</h2>

In [6]:
def crossing_budget_of(sampler, band=BAND):
    row = mean_energy.loc[sampler].dropna()
    row = row[row.index > 1]
    for k in row.index:
        if row.loc[k] < band and (row.loc[row.index >= k] < band).all():
            return int(k)
    return None


cross_fm = crossing_budget_of("fm_euler")
cross_ddpm = crossing_budget_of("ddpm_ancestral")
print(f"flow matching crosses at {cross_fm}, ancestral DDPM at {cross_ddpm}, "
      f"a ratio of {cross_ddpm / cross_fm:.0f}")
print(f"but at 200 steps: fm {mean_energy.loc['fm_euler', 200]:.5f} against "
      f"ddpm {mean_energy.loc['ddpm_ancestral', 200]:.5f}, "
      f"a ratio of {mean_energy.loc['ddpm_ancestral', 200] / mean_energy.loc['fm_euler', 200]:.1f}")

flow matching crosses at 8, ancestral DDPM at 32, a ratio of 4
but at 200 steps: fm 0.00147 against ddpm 0.00177, a ratio of 1.2


**The two loss lines.**

The diffusion loss regresses the network's output onto the noise that was added:
`loss = ((pred - eps) ** 2).mean()`, where `eps` is the Gaussian sample that was
used to corrupt `x0` up to level `t`.

The flow-matching loss regresses the same network's output onto a velocity:
`loss = ((vt - ut) ** 2).mean()`, where `ut` is `x1 - x0`, the constant velocity
along the straight line joining a noise point to a data point.

Same architecture, same parameter count, same optimizer, same schedule. Only the
regression target changes on one line of the training loop. Seeing that swap on a
2D MLP prepares you for module 3, where the same target swap sits inside an
equivariant message-passing network.

**The crossing budgets and the ratio.** Flow matching crosses the 0.01 band at 8
steps and ancestral DDPM at 32, a fourfold advantage.

**Why the ratio is meaningless without the band.** The two curves are not parallel,
so the advantage is a function of where you measure it. At the 0.01 band the ratio of
crossing budgets is 4. At a matched budget of 200 steps the two samplers are at
0.00147 and 0.00177, a ratio of 1.2, and at that point they are effectively
indistinguishable for any practical purpose. Quote "four times better" without the
band and a reader will reasonably assume you mean four times better in general, which
is a claim the data does not support at any budget above about 32.

This is the first appearance of a move that comes back twice. In X6 the number is
diversity and the missing qualifier is the fingerprint. In X8 the number is retention
and the missing qualifier is the definition of retention. In all three cases a single
figure is defensible only when accompanied by the thing it is relative to, and the
graded skill is noticing that the qualifier is missing rather than knowing the
particular answer.

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Answer summary</h2>

X1: crossing budget 32; energy distance about 0.0018 at 200 steps and about 0.041 at
8; the scatter plots would have located the failure two to four times later than the
metric does.

X2: two-sample floor about 0.0001, so the 0.01 band sits roughly a hundred times
above the metric's own noise; the confound is sampling stochasticity; holding it
fixed the gap widens, and removing the noise at a fixed 8-step budget costs a factor
of about 21.

X3: the diffusion loss regresses onto the added noise, the flow-matching loss onto
the velocity x1 minus x0; crossing budgets 8 and 32 for a fourfold ratio; the ratio
is defined only relative to the band, because at 200 steps the same two samplers
differ by a factor of 1.2.